# Day 3 | Lab 3.2: LangChain Runnables Deep-Dive (LCEL Mastery)

**Duration:** ~1.5 hours

**Scenario.** Customer-complaint orchestration pipeline — preserved from the GM source notebook. We push every LCEL primitive through real chains so trainees develop muscle memory for composing them in production.

**Learning Objectives.** By the end of this lab, you will be able to:
1. Compose chains with `RunnablePassthrough` to keep the input flowing alongside transformed output.
2. Use `RunnablePassthrough.assign()` to enrich inputs mid-pipeline (add metadata, computed fields).
3. Use `RunnableLambda` as the escape hatch from LCEL into ordinary Python.
4. Run independent branches concurrently with **`RunnableParallel`** — measure the real latency win.
5. Build conditional routing with **`RunnableBranch`** (if/elif/else inside LCEL).
6. Add production resilience with **`.with_fallbacks()`** and **`.with_retry()`** — and combine them safely.
7. Process inputs concurrently with **`.batch()`** for I/O-bound workloads.
8. Inject runtime config (e.g., `user_id`) into a chain via `RunnableConfig`.

---


## 1. Install Dependencies

In [ ]:
# Required packages for this lab — already installed in your local venv.
# To install standalone, uncomment the line(s) below:
# !pip install -q 'langchain>=1.0' 'langchain-core>=1.0' 'langchain-openai>=1.0' 'langchain-groq>=0.2'


In [ ]:
import textwrap

def wrap_print(text, width=80):
    """Prints text wrapped to a specified width."""
    print(textwrap.fill(text, width=width))
    print()

# Example usage with a long string
long_text = "This is a very long string that would normally require horizontal scrolling to read completely if it were just printed out directly to the console without any formatting. But with this helper function, it will be nicely wrapped!"

wrap_print(long_text)

This is a very long string that would normally require horizontal scrolling to
read completely if it were just printed out directly to the console without any
formatting. But with this helper function, it will be nicely wrapped!



## 2. API Key Configuration

In [ ]:
import os

# Local-venv pattern: load from .env if python-dotenv is available, otherwise rely on
# environment variables already set in your shell or venv activation script.
try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

# Verify keys are loaded (prints status, never prints actual values)
for key in ['OPENAI_API_KEY', 'GROQ_API_KEY']:
    status = '✅ Loaded' if os.environ.get(key) else '❌ MISSING'
    print(f'{key}: {status}')


## 3. Initialize LLMs

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_groq import ChatGroq

# Primary model — OpenAI gpt-4.1-mini (supports temperature)
llm_openai = ChatOpenAI(model="gpt-4.1-mini", temperature=0.3)

# Secondary model — Groq (open-source Llama 3.3)
llm_groq = ChatGroq(model="llama-3.3-70b-versatile", temperature=0.3)

# Quick smoke test
wrap_print(llm_openai.invoke("Say 'LLMs ready'").content)
wrap_print(llm_groq.invoke("Say 'LLMs ready'").content)

LLMs ready
LLMs ready


---
## 4. RunnablePassthrough — Pass Data Unchanged

### Business Scenario: Loan Application RAG Pattern
A retail bank's FAQ chatbot retrieves policy documents and combines them with the user's original question. `RunnablePassthrough` forwards the user question unchanged while other branches fetch context — this is the foundational RAG data-flow pattern.

> **When to use:** Whenever you need to carry the original input forward alongside transformed/enriched data.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableParallel

# Simulate a retriever — in production this would be a vector store lookup
def fake_retriever(question: str) -> str:
    """Simulates retrieving relevant bank policy docs based on question keywords."""
    policy_db = {
        "eligibility": "Home loan eligibility: min age 21, max 60. Min income ₹25,000/month. CIBIL score ≥ 700.",
        "interest": "Current home loan rates: 8.5% fixed, 8.25% floating. Processing fee: 0.5% of loan amount.",
        "documents": "Required docs: PAN card, Aadhaar, last 6 months bank statements, salary slips, property papers.",
    }
    for keyword, doc in policy_db.items():
        if keyword in question.lower():
            return doc
    return "No specific policy found. Please consult your branch relationship manager."


# RAG prompt — needs both 'context' (retrieved) and 'question' (original)
rag_prompt = ChatPromptTemplate.from_template(
    """You are a helpful banking assistant. Answer the customer's question using ONLY the provided context.
If the context doesn't cover the question, say so.

Context: {context}
Question: {question}

Answer:"""
)

# RunnableParallel feeds the same input to multiple branches:
#   - "context" branch: calls the retriever
#   - "question" branch: RunnablePassthrough() passes the input string unchanged
rag_chain = (
    RunnableParallel(
        context=lambda x: fake_retriever(x),   # retriever gets the question
        question=RunnablePassthrough()           # question passes through as-is
    )
    | rag_prompt
    | llm_openai
    | StrOutputParser()
)

# Test with different customer questions
for q in ["What is the eligibility for a home loan?",
          "What interest rate do you offer on home loans?",
          "What documents do I need to apply?"]:
    wrap_print(f"\n❓ {q}")
    wrap_print(f"💬 {rag_chain.invoke(q)}")
    print("-" * 60)

 ❓ What is the eligibility for a home loan?
💬 The eligibility for a home loan is: minimum age 21 years, maximum age 60
years, minimum income ₹25,000 per month, and a CIBIL score of 700 or above.
------------------------------------------------------------
 ❓ What interest rate do you offer on home loans?
💬 We offer home loans at an interest rate of 8.5% fixed and 8.25% floating.
------------------------------------------------------------
 ❓ What documents do I need to apply?
💬 You need to provide your PAN card, Aadhaar, last 6 months bank statements,
salary slips, and property papers to apply.
------------------------------------------------------------


> **Key Insight:** `RunnablePassthrough()` is not "doing nothing" — it's the critical glue that carries the original user input forward while other branches transform or enrich the data. You'll see this pattern everywhere in RAG systems.

---
## 5. RunnablePassthrough.assign() — Augment Data Mid-Pipeline

### Business Scenario: Insurance Claim Enrichment
An insurance company receives raw claim descriptions. Before the LLM generates a response, the pipeline needs to **add computed fields** (word count, urgency flag, timestamp) to the data — without losing the original fields.

`.assign()` adds new keys to a dict while keeping all existing keys intact.

In [ ]:
from langchain_core.runnables import RunnablePassthrough, RunnableLambda
from datetime import datetime

# Enrichment functions — pure Python logic we want to inject into the chain
def compute_urgency(data: dict) -> str:
    """Flag high-urgency claims based on keywords."""
    urgent_keywords = ["flood", "fire", "accident", "hospitalized", "emergency", "theft"]
    text = data["claim_text"].lower()
    return "HIGH" if any(kw in text for kw in urgent_keywords) else "NORMAL"

def compute_word_count(data: dict) -> int:
    return len(data["claim_text"].split())


# .assign() adds new keys to the dict flowing through the pipeline
enrichment_chain = RunnablePassthrough.assign(
    urgency=RunnableLambda(compute_urgency),
    word_count=RunnableLambda(compute_word_count),
    received_at=lambda _: datetime.now().isoformat()
)

# Now chain it into an LLM call
triage_prompt = ChatPromptTemplate.from_template(
    """You are an insurance claims triage assistant.

Claim from: {customer_name}
Policy: {policy_id}
Urgency: {urgency}
Description ({word_count} words): {claim_text}
Received: {received_at}

Generate a short triage summary (2-3 sentences) with recommended next steps."""
)

full_triage_chain = enrichment_chain | triage_prompt | llm_openai | StrOutputParser()

# Test with two claims — one urgent, one routine
claims = [
    {"customer_name": "Priya Sharma", "policy_id": "HI-2024-8831",
     "claim_text": "My house was damaged in the flood last week. Ground floor completely submerged. Furniture destroyed."},
    {"customer_name": "Rahul Mehra", "policy_id": "HI-2024-5520",
     "claim_text": "Requesting reimbursement for routine dental checkup and cleaning done on 15th January."}
]

for claim in claims:
    # First, see what .assign() produces
    enriched = enrichment_chain.invoke(claim)
    wrap_print(f"\n📋 {enriched['customer_name']} | Urgency: {enriched['urgency']} | Words: {enriched['word_count']}")

    # Then run the full chain
    result = full_triage_chain.invoke(claim)
    wrap_print(f"🤖 {result}")
    print("-" * 60)

 📋 Priya Sharma | Urgency: HIGH | Words: 15
🤖 Claim from Priya Sharma (Policy HI-2024-8831) reports severe flood damage with
the ground floor completely submerged and furniture destroyed. Given the high
urgency, immediate assessment by a claims adjuster is recommended to document
damages and initiate the claim process. Advise the claimant to provide photos
and any repair estimates as soon as possible.
------------------------------------------------------------
 📋 Rahul Mehra | Urgency: NORMAL | Words: 12
🤖 Claim from Rahul Mehra under policy HI-2024-5520 requests reimbursement for a
routine dental checkup and cleaning dated January 15th. Since this is a routine
dental service with normal urgency, verify coverage eligibility and receipt
validity before processing. Recommend forwarding the claim to the dental claims
team for standard review and reimbursement.
------------------------------------------------------------


> **Key Insight:** `.assign()` is perfect for mid-pipeline enrichment — adding metadata, computing flags, injecting timestamps — all while preserving the original data dict. Think of it as a non-destructive `dict.update()`.

---
## 6. RunnableLambda — Custom Python Logic in Chains

### Business Scenario: E-Commerce Review Processing
An online retailer wants to pre-process and post-process data around the LLM call: clean the input text, then parse and validate the LLM's JSON output. `RunnableLambda` wraps any Python function into a chain-compatible step.

In [ ]:
import json
import re
from langchain_core.runnables import RunnableLambda

# --- Pre-processing: clean raw review text ---
def clean_review(raw: str) -> str:
    """Strip HTML tags, excessive whitespace, normalize text."""
    text = re.sub(r'<[^>]+>', '', raw)       # remove HTML tags
    text = re.sub(r'\s+', ' ', text).strip()  # collapse whitespace
    return text[:500]                          # truncate to 500 chars for token efficiency

# --- Post-processing: parse and validate JSON from LLM ---
def parse_review_json(llm_output: str) -> dict:
    """Extract JSON from LLM response, handle markdown code fences."""
    # Strip markdown ```json ... ``` wrappers if present
    cleaned = re.sub(r'```json\s*', '', llm_output)
    cleaned = re.sub(r'```\s*', '', cleaned).strip()
    try:
        data = json.loads(cleaned)
        # Validate expected fields
        required = ["sentiment", "rating", "key_topics"]
        for field in required:
            if field not in data:
                data[field] = "MISSING"
        return data
    except json.JSONDecodeError:
        return {"error": "Failed to parse JSON", "raw": llm_output[:200]}


# Build the chain: clean → prompt → LLM → parse
review_prompt = ChatPromptTemplate.from_template(
    """Analyze this product review and return ONLY valid JSON (no extra text):

Review: {review}

Return this exact format:
{{
  "sentiment": "positive" | "negative" | "mixed",
  "rating": <1-5>,
  "key_topics": ["topic1", "topic2"],
  "summary": "<one sentence summary>"
}}"""
)

review_chain = (
    RunnableLambda(clean_review)              # Step 1: pre-process
    | (lambda text: {"review": text})          # Step 2: wrap into prompt variable
    | review_prompt                             # Step 3: format prompt
    | llm_openai                                # Step 4: call LLM
    | StrOutputParser()                         # Step 5: extract text
    | RunnableLambda(parse_review_json)         # Step 6: post-process & validate
)

# Test with messy real-world input
raw_reviews = [
    "<p>Absolutely <b>love</b> this laptop! Battery lasts all day, keyboard feels great. Only issue is the webcam quality is mediocre.   Highly recommend for   remote work.</p>",
    "<div>Terrible experience.  Shoes arrived damaged, wrong size, and customer support was unhelpful. <br/>Want a refund ASAP.</div>",
]

for raw in raw_reviews:
    result = review_chain.invoke(raw)
    wrap_print(f"\n📦 Result: {json.dumps(result, indent=2)}")
    print("-" * 50)

 📦 Result: {   "sentiment": "positive",   "rating": 4,   "key_topics": [
"battery life",     "keyboard",     "webcam quality",     "remote work"   ],
"summary": "The laptop is great for remote work with excellent battery life and
keyboard, but the webcam quality is mediocre." }
--------------------------------------------------
 📦 Result: {   "sentiment": "negative",   "rating": 1,   "key_topics": [
"product quality",     "customer service",     "order accuracy",     "refund"
],   "summary": "The customer had a terrible experience with damaged shoes,
incorrect size, and unhelpful customer support, requesting a refund
immediately." }
--------------------------------------------------


> **Key Insight:** `RunnableLambda` is your escape hatch from the LCEL world into regular Python. Use it for data cleaning, validation, format conversion, API calls, database writes — anything that isn't a prompt/LLM/parser.

---
## 7. RunnableParallel — Concurrent Multi-Analysis

### Business Scenario: Healthcare Patient Feedback Dashboard
A hospital collects patient feedback after discharge. The analytics team needs **three parallel analyses** on each feedback entry — sentiment, department identification, and action item extraction. Running them sequentially would triple latency; `RunnableParallel` runs all three concurrently.

In [ ]:
import time
from langchain_core.runnables import RunnableParallel

# Three independent analysis chains — each takes feedback text as input
sentiment_chain = (
    ChatPromptTemplate.from_template(
        "Rate the sentiment of this patient feedback as POSITIVE, NEGATIVE, or MIXED. "
        "Reply with just the label.\n\nFeedback: {feedback}"
    )
    | llm_openai | StrOutputParser()
)

department_chain = (
    ChatPromptTemplate.from_template(
        "Identify which hospital department this feedback is about. "
        "Choose from: Emergency, Cardiology, Orthopedics, Radiology, Billing, Nursing, General. "
        "Reply with just the department name.\n\nFeedback: {feedback}"
    )
    | llm_openai | StrOutputParser()
)

action_chain = (
    ChatPromptTemplate.from_template(
        "Extract 1-2 specific action items from this patient feedback. "
        "If none, say 'No action needed'.\n\nFeedback: {feedback}"
    )
    | llm_openai | StrOutputParser()
)

# Run all three concurrently
parallel_analysis = RunnableParallel(
    sentiment=sentiment_chain,
    department=department_chain,
    action_items=action_chain
)

# Test with real feedback
feedbacks = [
    "The ER staff was incredible — Dr. Mehta and the nurses responded within minutes of my arrival. "
    "However, the billing department took 3 weeks to process my insurance claim, which was frustrating.",

    "My knee replacement surgery went well, but post-op physiotherapy scheduling was chaotic. "
    "I had to call 4 times to get an appointment. The surgical team was excellent though."
]

for fb in feedbacks:
    start = time.time()
    result = parallel_analysis.invoke({"feedback": fb})
    elapsed = round(time.time() - start, 2)

    wrap_print(f"\n📝 Feedback: {fb[:80]}...")
    print(f"   Sentiment:    {result['sentiment']}")
    print(f"   Department:   {result['department']}")
    wrap_print(f"   Action Items: {result['action_items']}")
    print(f"   ⏱️  All 3 analyses completed in {elapsed}s (parallel)")
    print("-" * 60)

 📝 Feedback: The ER staff was incredible — Dr. Mehta and the nurses responded
within minutes ...
   Sentiment:    MIXED
   Department:   Emergency
   Action Items: 1. Improve the processing time of insurance claims in the
billing department.   2. Communicate more clearly with patients about expected
timelines for insurance claim processing.
   ⏱️  All 3 analyses completed in 0.82s (parallel)
------------------------------------------------------------
 📝 Feedback: My knee replacement surgery went well, but post-op physiotherapy
scheduling was ...
   Sentiment:    MIXED
   Department:   Orthopedics
   Action Items: 1. Improve the scheduling process for post-operative
physiotherapy to reduce the need for multiple patient calls.   2. Implement a
more efficient communication system to confirm physiotherapy appointments
promptly.
   ⏱️  All 3 analyses completed in 1.04s (parallel)
------------------------------------------------------------


> **Key Insight:** `RunnableParallel` doesn't just organize code — it delivers real latency savings. Three 1-second LLM calls run in ~1s instead of ~3s. Essential when your pipeline has independent branches.

---
## 8. RunnableSequence — Explicit Sequential Composition

### Business Scenario: Sales Lead Qualification Pipeline
A B2B SaaS company qualifies inbound leads through a 3-step pipeline: (1) extract company info, (2) score the lead, (3) draft a personalized outreach email. `RunnableSequence` makes the sequential flow explicit.

> The pipe operator `|` creates `RunnableSequence` objects behind the scenes. Using `RunnableSequence(...)` directly is useful when building chains **programmatically** (e.g., from config files).

In [ ]:
from langchain_core.runnables import RunnableSequence, RunnableLambda

# Step 1: Extract company info from raw lead description
extract_prompt = ChatPromptTemplate.from_template(
    """Extract key info from this sales lead inquiry. Return ONLY valid JSON:
{{
  "company": "<name>",
  "industry": "<sector>",
  "size": "<startup/mid-market/enterprise>",
  "need": "<what they're looking for>"
}}

Lead inquiry: {inquiry}"""
)
extract_chain = extract_prompt | llm_openai | StrOutputParser()

# Step 2: Score the lead based on extracted info
score_prompt = ChatPromptTemplate.from_template(
    """Based on this extracted lead info, assign a qualification score (1-10) and priority.
Scoring criteria: enterprise + clear need = 8-10, mid-market = 5-7, startup/vague = 1-4.

Lead info: {lead_info}

Return ONLY: "Score: X/10 | Priority: HIGH/MEDIUM/LOW | Reason: <brief>"""
)

def bridge_extract_to_score(extracted_text: str) -> dict:
    """Bridge function: output of step 1 becomes input for step 2."""
    return {"lead_info": extracted_text}

score_chain = RunnableLambda(bridge_extract_to_score) | score_prompt | llm_openai | StrOutputParser()

# Step 3: Draft outreach email
email_prompt = ChatPromptTemplate.from_template(
    """Based on this lead scoring, draft a short personalized outreach email (3-4 sentences).
Be professional but warm. Mention their specific need.

Lead score: {score_info}"""
)

def bridge_score_to_email(score_text: str) -> dict:
    return {"score_info": score_text}

email_chain = RunnableLambda(bridge_score_to_email) | email_prompt | llm_openai | StrOutputParser()

# --- Compose all 3 steps using RunnableSequence ---
lead_pipeline = RunnableSequence(extract_chain, score_chain, email_chain)

# Test
inquiry = (
    "Hi, I'm the VP of Engineering at FinServe Corp (2000+ employees). We're looking for an "
    "AI-powered document processing solution to automate our KYC compliance workflows. "
    "Currently processing 10,000 applications per month manually. Need to reduce turnaround from 5 days to under 24 hours."
)

print("🔄 Running 3-step lead qualification pipeline...\n")
result = lead_pipeline.invoke({"inquiry": inquiry})
wrap_print(f"📧 Final outreach email:\n{result}")

🔄 Running 3-step lead qualification pipeline...

📧 Final outreach email: Subject: Helping [Company Name] Enhance Compliance
Workflow Efficiency with AI  Hi [Recipient’s Name],  I noticed that [Company
Name] is focused on improving compliance workflow efficiency, and I believe our
AI-powered automation solutions can help streamline your processes while
ensuring accuracy. I’d love to explore how we can support your team in meeting
these goals effectively. Would you be open to a brief conversation next week?
Best regards,   [Your Name]


> **Key Insight:** `RunnableSequence(step1, step2, step3)` is equivalent to `step1 | step2 | step3`. Use the explicit form when you're building chains dynamically — e.g., assembling steps from a config file or user selections.

---
## 9. RunnableBranch — Conditional Routing

### Business Scenario: Customer Support Query Router
A bank's AI support system receives diverse customer queries — billing disputes, loan inquiries, fraud reports, general questions. Each category needs a **specialized prompt** with domain-specific instructions. `RunnableBranch` routes the query to the right chain based on conditions.

In [ ]:
from langchain_core.runnables import RunnableBranch, RunnableLambda

# --- Category detection functions ---
def is_fraud(query: dict) -> bool:
    keywords = ["fraud", "unauthorized", "stolen", "suspicious", "hacked", "phishing"]
    return any(kw in query["question"].lower() for kw in keywords)

def is_loan(query: dict) -> bool:
    keywords = ["loan", "emi", "mortgage", "interest rate", "pre-approval", "refinance"]
    return any(kw in query["question"].lower() for kw in keywords)

def is_billing(query: dict) -> bool:
    keywords = ["bill", "charge", "fee", "statement", "transaction", "overcharged"]
    return any(kw in query["question"].lower() for kw in keywords)

# --- Specialized chains for each category ---
fraud_chain = (
    ChatPromptTemplate.from_template(
        """🚨 FRAUD ALERT HANDLER: You are a fraud specialist. Be urgent but reassuring.
Immediately advise the customer to secure their account.
Provide specific steps: block card, change password, file dispute.

Customer query: {question}"""
    ) | llm_openai | StrOutputParser()
)

loan_chain = (
    ChatPromptTemplate.from_template(
        """🏦 LOAN ADVISOR: You are a loan specialist. Be informative and helpful.
Cover: eligibility basics, current rate ranges, next steps to apply.

Customer query: {question}"""
    ) | llm_openai | StrOutputParser()
)

billing_chain = (
    ChatPromptTemplate.from_template(
        """💳 BILLING SUPPORT: You are a billing specialist. Be precise and empathetic.
Explain the charge, offer to investigate, and mention dispute resolution options.

Customer query: {question}"""
    ) | llm_openai | StrOutputParser()
)

general_chain = (
    ChatPromptTemplate.from_template(
        """🏦 GENERAL BANKING SUPPORT: You are a helpful banking assistant.
Answer the question clearly. If you're unsure, suggest visiting a branch.

Customer query: {question}"""
    ) | llm_openai | StrOutputParser()
)

# --- RunnableBranch: if/elif/else routing ---
router = RunnableBranch(
    (is_fraud,   fraud_chain),     # if fraud → fraud specialist
    (is_loan,    loan_chain),      # elif loan → loan advisor
    (is_billing, billing_chain),   # elif billing → billing support
    general_chain                   # else → general support (default/fallback)
)

# Test with different query types
test_queries = [
    "I see an unauthorized transaction of ₹45,000 on my credit card from yesterday!",
    "What's the current interest rate for a home loan? I want to refinance.",
    "There's a ₹500 charge on my statement I don't recognize. Can you explain?",
    "What are your branch timings on weekends?"
]

for q in test_queries:
    wrap_print(f"\n❓ {q}")
    result = router.invoke({"question": q})
    wrap_print(f"💬 {result[:200]}...")
    print("=" * 60)

 ❓ I see an unauthorized transaction of ₹45,000 on my credit card from
yesterday!

💬 🚨 URGENT: Please act immediately to secure your account!  1. **Block your
credit card right now** to prevent any further unauthorized transactions. You
can do this via your bank’s app or by calling cu...

 ❓ What's the current interest rate for a home loan? I want to refinance.

💬 Hello! I'd be happy to help you with refinancing your home loan.  **Current
Interest Rate Ranges:**   Interest rates for home loan refinancing typically
vary based on factors like your credit score, l...

 ❓ There's a ₹500 charge on my statement I don't recognize. Can you explain?

💬 Hello, I understand your concern about the ₹500 charge on your statement. This
charge may be related to a recent transaction or service linked to your account.
I’d be happy to investigate this further...

 ❓ What are your branch timings on weekends?

💬 Our branch timings on weekends may vary by location. Generally, branches are
open from 9:00 AM 

> **Key Insight:** `RunnableBranch` gives you `if/elif/else` logic inside LCEL chains. The conditions are Python callables (functions or lambdas) that return `True/False`. The **last argument** (no condition) is the default fallback. In production, you'd often use an LLM-based classifier instead of keyword matching.

---
## 10. `.with_fallbacks()` — Provider Failover

### Business Scenario: Production Chatbot Reliability
A fintech company's customer-facing chatbot primarily uses OpenAI, but needs **automatic failover** to Groq (Llama) if OpenAI's API is down or throws an error. `.with_fallbacks()` chains backup providers transparently.

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_groq import ChatGroq

# Primary: OpenAI (intentionally using a WRONG model name to simulate failure)
primary_llm = ChatOpenAI(model="gpt-4.1-mini-WRONG-NAME", temperature=0.3)

# Fallback: Groq (will catch the error and use this instead)
fallback_llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0.3)

# Chain with automatic failover
resilient_llm = primary_llm.with_fallbacks([fallback_llm])

# Build a chain using the resilient LLM
resilient_chain = (
    ChatPromptTemplate.from_template(
        "You are a financial advisor. Briefly answer: {question}"
    )
    | resilient_llm
    | StrOutputParser()
)

# This will fail on OpenAI (bad model name), then automatically retry with Groq
print("🔄 Calling with intentionally broken primary (OpenAI) + working fallback (Groq)...\n")
result = resilient_chain.invoke({"question": "What's the 50-30-20 budgeting rule?"})
wrap_print(f"💬 {result}")
print("\n✅ Fallback kicked in seamlessly — the user never sees the primary failure.")

🔄 Calling with intentionally broken primary (OpenAI) + working fallback (Groq)...

💬 The 50-30-20 budgeting rule is a simple guideline for allocating your income.
It suggests that:  * 50% of your income should go towards necessary expenses
(housing, utilities, food, transportation, and minimum payments on debts) * 30%
towards discretionary spending (entertainment, hobbies, travel) * 20% towards
saving and debt repayment (emergency fund, retirement savings, paying off high-
interest debts)  This rule helps you strike a balance between enjoying your life
today and securing your financial future.


✅ Fallback kicked in seamlessly — the user never sees the primary failure.


In [ ]:
# --- Now demonstrate with WORKING primary (no fallback needed) ---
healthy_llm = ChatOpenAI(model="gpt-4.1-mini", temperature=0.3)
healthy_resilient = healthy_llm.with_fallbacks([fallback_llm])

healthy_chain = (
    ChatPromptTemplate.from_template("You are a financial advisor. Briefly answer: {question}")
    | healthy_resilient
    | StrOutputParser()
)

print("🔄 Calling with healthy primary (OpenAI) — fallback is idle...\n")
result = healthy_chain.invoke({"question": "What's the 50-30-20 budgeting rule?"})
wrap_print(f"💬 {result}")
print("\n✅ Primary succeeded — fallback was available but not needed.")

🔄 Calling with healthy primary (OpenAI) — fallback is idle...

💬 The 50-30-20 budgeting rule is a simple guideline for managing your income:
allocate 50% to needs (essentials like housing, food, bills), 30% to wants (non-
essentials like dining out, entertainment), and 20% to savings and debt
repayment.


✅ Primary succeeded — fallback was available but not needed.


> **Key Insight:** `.with_fallbacks([backup1, backup2])` is production-critical. You can chain multiple fallbacks (OpenAI → Groq → local model). The user never sees the failure — the system automatically retries with the next provider.

---
## 11. `.with_retry()` — Handle Transient Failures

### Business Scenario: Batch Processing During Peak Hours
An e-commerce platform runs nightly batch analysis on customer reviews. During peak API hours, rate limit errors (`429`) and timeouts are common. `.with_retry()` handles these transient failures automatically with exponential backoff.

In [ ]:
# .with_retry() adds automatic retry with exponential backoff
retry_llm = llm_openai.with_retry(
    stop_after_attempt=3,                            # max 3 attempts
    wait_exponential_jitter=True                     # exponential backoff with jitter
)

retry_chain = (
    ChatPromptTemplate.from_template(
        "Classify this product review as POSITIVE, NEGATIVE, or NEUTRAL. Reply with just the label.\n\n"
        "Review: {review}"
    )
    | retry_llm
    | StrOutputParser()
)

# This will work normally — retry logic is invisible unless there's a failure
result = retry_chain.invoke({"review": "Great product, fast delivery, would buy again!"})
print(f"Result: {result}")
print("\n✅ .with_retry() is a silent safety net — kicks in only when transient errors occur.")

Result: POSITIVE

✅ .with_retry() is a silent safety net — kicks in only when transient errors occur.


In [ ]:
# --- Combining retry + fallbacks (belt AND suspenders) ---
# This is the production-grade pattern: retry first, then fallback
production_llm = (
    ChatOpenAI(model="gpt-4.1-mini", temperature=0.3)
    .with_retry(stop_after_attempt=2)
    .with_fallbacks(
        [ChatGroq(model="llama-3.3-70b-versatile", temperature=0.3).with_retry(stop_after_attempt=2)]
    )
)

production_chain = (
    ChatPromptTemplate.from_template("Briefly explain: {topic}")
    | production_llm
    | StrOutputParser()
)

result = production_chain.invoke({"topic": "what is compound interest"})
wrap_print(f"💬 {result}")
print("✅ Production pattern: retry(OpenAI, 2x) → fallback → retry(Groq, 2x)")

💬 Compound interest is the interest calculated on the initial principal as well
as on the accumulated interest from previous periods. This means you earn
interest on both your original investment and the interest that has been added
over time, leading to exponential growth of the investment.

✅ Production pattern: retry(OpenAI, 2x) → fallback → retry(Groq, 2x)


> **Production Pattern:** Always combine `.with_retry()` AND `.with_fallbacks()` for customer-facing systems. Retry handles transient errors (rate limits, timeouts); fallbacks handle persistent failures (provider outage).

---
## 12. `.batch()` — Process Multiple Inputs Concurrently

### Business Scenario: Bulk Insurance Claim Classification
An insurance company receives hundreds of claims daily. Instead of processing them one by one (sequential `.invoke()`), `.batch()` sends multiple inputs through the chain concurrently.

In [ ]:
# Classification chain
classify_chain = (
    ChatPromptTemplate.from_template(
        """Classify this insurance claim into ONE category:
AUTO, HEALTH, PROPERTY, LIFE, TRAVEL.
Reply with just the category.

Claim: {claim}"""
    )
    | llm_openai
    | StrOutputParser()
)

# Batch of claims to process
claims_batch = [
    {"claim": "Rear-ended at a traffic signal. Bumper and tail lights damaged. Other driver at fault."},
    {"claim": "Emergency appendectomy surgery. 3 days hospitalization. Total bill ₹2.8 lakhs."},
    {"claim": "Kitchen fire caused by gas leak. Significant damage to cabinets and appliances."},
    {"claim": "Policyholder passed away due to cardiac arrest. Family filing death benefit claim."},
    {"claim": "Flight cancelled due to airline strike. Hotel and rebooking costs of ₹35,000."},
    {"claim": "Laptop stolen from hotel room during business trip to Singapore."},
]

# --- Sequential processing (baseline) ---
start = time.time()
sequential_results = [classify_chain.invoke(c) for c in claims_batch]
seq_time = round(time.time() - start, 2)

# --- Batch processing (concurrent) ---
start = time.time()
batch_results = classify_chain.batch(claims_batch)
batch_time = round(time.time() - start, 2)

# Compare
print("📊 RESULTS:")
for claim, category in zip(claims_batch, batch_results):
    print(f"  {category:10s} ← {claim['claim'][:80]}...")

print(f"\n⏱️  Sequential: {seq_time}s | Batch: {batch_time}s | Speedup: {round(seq_time/max(batch_time,0.01), 1)}x")

📊 RESULTS:
  AUTO       ← Rear-ended at a traffic signal. Bumper and tail lights damaged. Other driver at ...
  HEALTH     ← Emergency appendectomy surgery. 3 days hospitalization. Total bill ₹2.8 lakhs....
  PROPERTY   ← Kitchen fire caused by gas leak. Significant damage to cabinets and appliances....
  LIFE       ← Policyholder passed away due to cardiac arrest. Family filing death benefit clai...
  TRAVEL     ← Flight cancelled due to airline strike. Hotel and rebooking costs of ₹35,000....
  PROPERTY   ← Laptop stolen from hotel room during business trip to Singapore....

⏱️  Sequential: 2.7s | Batch: 0.6s | Speedup: 4.5x


> **Key Insight:** `.batch()` processes multiple inputs concurrently through the same chain. For I/O-bound operations (API calls), this delivers significant speedup. Use `config={"max_concurrency": 5}` to control parallelism and avoid rate limits.

---
## 13. Putting It All Together — End-to-End Pipeline

### Business Scenario: E-Commerce Order Complaint Resolution
An e-commerce company's complaint pipeline combines **multiple Runnable types** in a single flow:

1. **RunnableLambda** — Pre-process raw complaint text
2. **RunnablePassthrough.assign()** — Enrich with metadata (urgency, timestamp)
3. **RunnableBranch** — Route to refund, replacement, or general support chain
4. **RunnableParallel** — Generate response + log summary concurrently
5. **`.with_retry()`** — Handle transient API failures

In [ ]:
from langchain_core.runnables import (
    RunnablePassthrough, RunnableLambda, RunnableParallel, RunnableBranch
)

# ── Step 1: Pre-process ──
def preprocess_complaint(raw: dict) -> dict:
    """Clean text and add metadata."""
    text = raw["complaint_text"].strip()
    return {
        "complaint_text": text,
        "customer_name": raw.get("customer_name", "Unknown"),
        "order_id": raw.get("order_id", "N/A"),
    }

# ── Step 2: Enrich with urgency + timestamp ──
def detect_urgency(data: dict) -> str:
    urgent_kw = ["refund", "broken", "damaged", "wrong item", "never arrived", "scam"]
    return "HIGH" if any(kw in data["complaint_text"].lower() for kw in urgent_kw) else "NORMAL"

# ── Step 3: Routing conditions ──
def wants_refund(data: dict) -> bool:
    return any(kw in data["complaint_text"].lower() for kw in ["refund", "money back", "cancel"])

def wants_replacement(data: dict) -> bool:
    return any(kw in data["complaint_text"].lower() for kw in ["replace", "exchange", "wrong item", "damaged"])

# ── Specialized response chains ──
refund_chain = (
    ChatPromptTemplate.from_template(
        """You are a refund specialist for an e-commerce company.
Customer: {customer_name} | Order: {order_id} | Urgency: {urgency}
Complaint: {complaint_text}

Draft a empathetic response confirming the refund process. Include timeline (5-7 business days)."""
    ) | llm_openai.with_retry(stop_after_attempt=2) | StrOutputParser()
)

replacement_chain = (
    ChatPromptTemplate.from_template(
        """You are a replacement/exchange specialist.
Customer: {customer_name} | Order: {order_id} | Urgency: {urgency}
Complaint: {complaint_text}

Draft a response offering a replacement with express shipping. Apologize for the inconvenience."""
    ) | llm_openai.with_retry(stop_after_attempt=2) | StrOutputParser()
)

general_support_chain = (
    ChatPromptTemplate.from_template(
        """You are a general customer support agent.
Customer: {customer_name} | Order: {order_id} | Urgency: {urgency}
Complaint: {complaint_text}

Draft a helpful response. Ask clarifying questions if needed."""
    ) | llm_openai.with_retry(stop_after_attempt=2) | StrOutputParser()
)

# ── Log summary chain (runs in parallel with response) ──
log_chain = (
    ChatPromptTemplate.from_template(
        """Create a 1-line internal log entry for this complaint.
Format: [URGENCY] Order#ORDER_ID — CATEGORY — brief summary

Customer: {customer_name} | Order: {order_id} | Urgency: {urgency}
Complaint: {complaint_text}"""
    ) | llm_openai | StrOutputParser()
)

# ── Assemble the full pipeline ──
complaint_pipeline = (
    # Step 1: Pre-process
    RunnableLambda(preprocess_complaint)
    # Step 2: Enrich
    | RunnablePassthrough.assign(
        urgency=RunnableLambda(detect_urgency),
        received_at=lambda _: datetime.now().isoformat()
    )
    # Step 3+4: Route AND generate log in parallel
    | RunnableParallel(
        response=RunnableBranch(
            (wants_refund, refund_chain),
            (wants_replacement, replacement_chain),
            general_support_chain
        ),
        log_entry=log_chain
    )
)

wrap_print("✅ Full complaint pipeline assembled. Testing...")

✅ Full complaint pipeline assembled. Testing...


In [ ]:
# --- Test with 3 different complaint types ---
test_complaints = [
    {
        "customer_name": "Anita Desai",
        "order_id": "ORD-2024-88321",
        "complaint_text": "I want a full refund. The headphones I received are completely broken — one earcup is cracked and there's no sound from the left side."
    },
    {
        "customer_name": "Vikram Patel",
        "order_id": "ORD-2024-77102",
        "complaint_text": "I received the wrong item! I ordered a blue formal shirt (size L) but got a red t-shirt (size S). Please replace it."
    },
    {
        "customer_name": "Sneha Rao",
        "order_id": "ORD-2024-99450",
        "complaint_text": "My order has been showing 'out for delivery' for the last 4 days. Can someone check what's happening with the courier?"
    }
]

for complaint in test_complaints:
    result = complaint_pipeline.invoke(complaint)
    print(f"\n{'=' * 70}")
    print(f"👤 {complaint['customer_name']} | 📦 {complaint['order_id']}")
    wrap_print(f"📋 Complaint: {complaint['complaint_text']}")
    wrap_print(f"\n📝 Log: {result['log_entry']}")
    wrap_print(f"\n💬 Response:\n{result['response']}")


👤 Anita Desai | 📦 ORD-2024-88321
📋 Complaint: I want a full refund. The headphones I received are completely
broken — one earcup is cracked and there's no sound from the left side.

 📝 Log: [HIGH] Order#ORD-2024-88321 — Product Defect — Headphones received
broken with cracked earcup and no sound on left side, customer requests full
refund.

 💬 Response: Subject: Refund Process for Your Order ORD-2024-88321  Dear Anita,
I’m very sorry to hear that the headphones you received arrived damaged and are
not functioning properly. I completely understand how disappointing this must
be, especially when you were expecting a fully working product.  Please rest
assured that we are processing a full refund for your order ORD-2024-88321. You
can expect the refund to be credited back to your original payment method within
5-7 business days.  If you have any further questions or need assistance, please
don’t hesitate to reach out. We appreciate your understanding and thank you for
giving us the oppor

---
## 14. Quick Reference: Runnable Types Cheatsheet

| Runnable | Import | One-Liner | When to Use |
|----------|--------|-----------|-------------|
| `RunnablePassthrough()` | `from langchain_core.runnables import RunnablePassthrough` | Pass input unchanged | RAG — carry question alongside retrieved context |
| `.assign(key=fn)` | Method on `RunnablePassthrough` | Add computed fields to dict | Enrich data mid-pipeline without losing existing keys |
| `RunnableLambda(fn)` | `from langchain_core.runnables import RunnableLambda` | Wrap Python function | Custom transforms, validation, I/O, formatting |
| `RunnableParallel(a=chain1, b=chain2)` | `from langchain_core.runnables import RunnableParallel` | Run branches concurrently | Independent analyses on same input |
| `RunnableSequence(s1, s2, s3)` | `from langchain_core.runnables import RunnableSequence` | Explicit sequential chain | Programmatic chain assembly (same as `s1 \| s2 \| s3`) |
| `RunnableBranch((cond, chain), ..., default)` | `from langchain_core.runnables import RunnableBranch` | if/elif/else routing | Route queries to specialized handlers |
| `chain.with_fallbacks([backup])` | Method on any Runnable | Auto failover | Provider redundancy (OpenAI → Groq) |
| `chain.with_retry(stop_after_attempt=3)` | Method on any Runnable | Auto retry with backoff | Handle rate limits, transient API errors |
| `chain.batch([input1, input2, ...])` | Method on any Runnable | Concurrent multi-input | Bulk processing documents, emails, tickets |

---

## 9. Conclusion & Key Takeaways

| Primitive | When to reach for it |
|---|---|
| `RunnablePassthrough()` | Carry the input through to a later step that needs it |
| `RunnablePassthrough.assign(field=fn)` | Enrich the input dict mid-pipeline (metadata, computed fields) |
| `RunnableLambda(fn)` | Escape hatch from LCEL into ordinary Python — wrap any function |
| `RunnableParallel({a: ch1, b: ch2})` | Independent branches; concurrent execution; measurable latency win |
| `RunnableSequence(s1, s2, s3)` | Same as `s1 \| s2 \| s3` — useful when constructing chains dynamically |
| `RunnableBranch((cond, ch), default)` | Conditional routing inside LCEL — if/elif/else |
| `.with_fallbacks([backup])` | Production resilience — when primary fails, fall through to backup |
| `.with_retry(max_attempts=3)` | Retry on transient errors with exponential backoff |
| `.batch(inputs)` | Concurrent processing of N inputs through the same chain |

## 10. Stretch Exercise (Optional)
1. **Routing LLM.** Replace a `RunnableBranch` with an LLM-as-router that classifies the input first, then routes to one of 3 specialized chains.
2. **Belt + suspenders.** Combine `.with_retry(max_attempts=3)` with `.with_fallbacks([backup_llm])` and verify the retry happens BEFORE the fallback.
3. **Config injection.** Pass a `user_id` through `chain.invoke(..., config={'configurable': {'user_id': 'U-123'}})` and have a `RunnableLambda` log it as part of structured logging.
4. **Adaptive batching.** Group 100 items into batches of 10 with `chain.batch(group, config={'max_concurrency': 5})` — measure throughput vs sequential.
5. **Streaming a parallel chain.** Use `RunnableParallel` with three branches and stream each branch independently using `.astream_events()`.

---

**Next Lab:** Lab 3.3 — Tool Calling Agents with LangChain v1 🛠️

---

## Interview Preparation

---

**Q1. When use `RunnablePassthrough.assign()` vs `RunnableParallel`?**

*Hint:* Enrich-and-keep vs branch-and-collect.

*Answer sketch:* `assign()` adds new keys to a dict that flows downstream — same input + new fields. `RunnableParallel` runs independent branches and produces a dict where each KEY is a branch's output (input is replaced, not enriched). Use `assign` when later steps still need the original input alongside the new field.

---

**Q2. What happens with concurrency under `RunnableParallel` — is it threaded or async?**

*Hint:* `asyncio.gather` under the hood.

*Answer sketch:* `RunnableParallel` uses `asyncio.gather` for `.ainvoke`/`.abatch` and a `ThreadPoolExecutor` for `.invoke`/`.batch`. For LLM API calls (I/O-bound) you get true concurrency in both modes — typical 2–4× wall-clock speedup for 3 branches.

---

**Q3. Difference between `RunnableLambda(fn)` and a Python function passed in directly?**

*Hint:* Auto-coercion.

*Answer sketch:* When you pipe a plain function (`some_fn | llm`), LangChain auto-wraps it in `RunnableLambda`. The explicit form is identical in behaviour but lets you set `name=` for tracing, `func_kwargs=`, or use it inside non-pipe contexts (e.g., as a value in `RunnableParallel`).

---

**Q4. When would `RunnableBranch` be the wrong tool — and what to use instead?**

*Hint:* When the routing condition itself needs an LLM.

*Answer sketch:* `RunnableBranch` takes deterministic predicates. If the routing decision is fuzzy ("is this complaint billing-related, technical, or fraud?"), use an LLM-as-router: a small classification chain whose output keys into a `RunnableBranch` — or directly returns the chain to invoke. Don't try to embed regex spaghetti in the branch predicates.

---

**Q5. `.with_fallbacks()` vs `.with_retry()` — what's each for?**

*Hint:* Provider failure vs transient error.

*Answer sketch:* `.with_retry()` re-runs the SAME chain on transient errors (rate limits, timeouts) with exponential backoff. `.with_fallbacks()` switches to a DIFFERENT chain after persistent failure. Production pattern: retry first, then fall back — `chain.with_retry(max_attempts=3).with_fallbacks([backup_chain])`.

---

**Q6. How does `.batch()` differ from a list comprehension calling `.invoke()`?**

*Hint:* Concurrency.

*Answer sketch:* A list comprehension runs invocations sequentially. `.batch(inputs, config={'max_concurrency': 5})` runs them concurrently (capped at 5) using thread/async pools. For I/O-bound LLM calls, this is a 5× speedup at no extra cost. For CPU-bound or rate-limit-sensitive calls, set `max_concurrency` lower.

---

**Q7. Show how to inject runtime config (e.g., a `user_id`) into a Runnable chain.**

*Hint:* `RunnableConfig` + `config={'configurable': {...}}` + `chain.with_config(...)`.

*Answer sketch:* Pass `config={'configurable': {'user_id': 'U-123', 'tenant': 'acme'}}` at invocation. Inside the chain, a `RunnableLambda` can accept a second arg `config: RunnableConfig` and read `config['configurable']['user_id']`. This is how `RunnableWithMessageHistory` reads `session_id` per-call.
